# SupplyShield AI — Final Risk Scoring Engine

This notebook integrates upstream NLP, anomaly, supplier, market, and WebShield signals into a unified and explainable 0–100 risk score.

The scoring engine is deterministic, configurable, robust to missing signals, and designed for downstream dashboard/API consumption.

In [20]:
# ============================================================
# SUPPLYSHIELD AI — FINAL RISK SCORING ENGINE
# CELL 2 — Imports & Global Configuration
# ============================================================

from __future__ import annotations

import json
import math
import re
import warnings
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# -----------------------------
# Display configuration
# -----------------------------
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

# -----------------------------
# Project paths
# -----------------------------
PROJECT_ROOT = Path.cwd()

# If notebook is executed from member2/notebooks,
# resolve project root automatically.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Search upwards for the actual project root.
for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    if (parent / "member2").exists():
        PROJECT_ROOT = parent
        break

DATA_PROCESSED = PROJECT_ROOT / "member2" / "data" / "processed"
DATA_OUTPUT = PROJECT_ROOT / "member2" / "data" / "processed" / "risk_scoring"
DATA_OUTPUT.mkdir(parents=True, exist_ok=True)

print("=" * 75)
print("SUPPLYSHIELD AI — FINAL RISK SCORING ENGINE")
print("=" * 75)
print(f"Project root     : {PROJECT_ROOT}")
print(f"Processed data   : {DATA_PROCESSED}")
print(f"Risk output dir  : {DATA_OUTPUT}")
print("=" * 75)

SUPPLYSHIELD AI — FINAL RISK SCORING ENGINE
Project root     : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI
Processed data   : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed
Risk output dir  : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\risk_scoring


## 1. Locate and Validate the Upstream Dataset

The scoring engine consumes the processed dataset produced by the preceding NLP/risk-engine stage.

In [21]:
# ============================================================
# CELL 4 — Robust Upstream Dataset Discovery
# ============================================================

def discover_input_files(base_dir: Path) -> List[Path]:
    """
    Discover candidate JSON/CSV/Parquet datasets from the
    processed-data directory.

    Risk-scoring outputs are excluded to avoid accidentally
    feeding the engine its own previous output.
    """
    
    if not base_dir.exists():
        return []

    candidates = []

    for pattern in ["*.json", "*.csv", "*.parquet"]:
        candidates.extend(base_dir.rglob(pattern))

    excluded_terms = {
        "risk_scoring",
        "final_risk",
        "supplier_risk",
        "risk_output"
    }

    filtered = []

    for path in candidates:
        lower = str(path).lower()

        if any(term in lower for term in excluded_terms):
            continue

        if path.name.lower() in {
            ".gitkeep",
            "package-lock.json"
        }:
            continue

        filtered.append(path)

    return sorted(
        filtered,
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )


candidate_files = discover_input_files(DATA_PROCESSED)

print("Candidate upstream datasets:")
print("-" * 75)

if not candidate_files:
    print("No candidate dataset found.")
else:
    for idx, path in enumerate(candidate_files[:30], start=1):
        size_kb = path.stat().st_size / 1024
        modified = datetime.fromtimestamp(
            path.stat().st_mtime
        ).strftime("%Y-%m-%d %H:%M:%S")

        print(
            f"{idx:02d}. {path.name:<45} "
            f"{size_kb:>10.2f} KB   {modified}"
        )

print("-" * 75)

Candidate upstream datasets:
---------------------------------------------------------------------------
01. unified_supply_data.parquet                       231.11 KB   2026-08-20 23:22:08
02. unified_supply_data.csv                           460.00 KB   2026-08-20 23:22:07
03. unified_supply_data.json                         1635.64 KB   2026-08-20 23:22:07
---------------------------------------------------------------------------


In [29]:
# ============================================================
# CELL 5 — Intelligent Dataset Loader
# ============================================================

def load_dataset(path: Path) -> pd.DataFrame:
    """
    Load JSON, CSV or Parquet datasets safely.
    """

    suffix = path.suffix.lower()

    if suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            raw = json.load(f)

        if isinstance(raw, list):
            df = pd.DataFrame(raw)

        elif isinstance(raw, dict):
            # Handle common wrapped dataset structures.
            possible_keys = [
                "data",
                "records",
                "results",
                "items",
                "dataset"
            ]

            records = None

            for key in possible_keys:
                if key in raw and isinstance(raw[key], list):
                    records = raw[key]
                    break

            if records is not None:
                df = pd.DataFrame(records)
            else:
                df = pd.DataFrame([raw])

        else:
            raise ValueError(
                f"Unsupported JSON structure in {path.name}"
            )

    elif suffix == ".csv":
        df = pd.read_csv(path)

    elif suffix == ".parquet":
        df = pd.read_parquet(path)

    else:
        raise ValueError(
            f"Unsupported file type: {suffix}"
        )

    return df


# ------------------------------------------------------------
# Prefer known upstream filename if available
# ------------------------------------------------------------

preferred_names = [
    "unified_supply_data.json",
    "nlp_risk_dataset.json",
    "nlp_enriched_supply_data.json",
    "supplier_risk_dataset.json",
    "feature_engineered_supply_data.json"
]

input_path = None

for name in preferred_names:
    candidate = DATA_PROCESSED / name

    if candidate.exists():
        input_path = candidate
        break


# ------------------------------------------------------------
# Otherwise select the most promising dataset automatically
# ------------------------------------------------------------

if input_path is None and candidate_files:

    scored_candidates = []

    important_terms = [
        "unified",
        "nlp",
        "risk",
        "feature",
        "supply"
    ]

    for path in candidate_files:

        filename = path.name.lower()

        score = sum(
            1 for term in important_terms
            if term in filename
        )

        scored_candidates.append(
            (score, path)
        )

    scored_candidates.sort(
        key=lambda x: (x[0], x[1].stat().st_mtime),
        reverse=True
    )

    input_path = scored_candidates[0][1]


if input_path is None:
    raise FileNotFoundError(
        f"""
No suitable upstream dataset was found.

Expected processed directory:
{DATA_PROCESSED}

Run Notebook 04 first and make sure its
processed output is saved inside member2/data/processed/.
"""
    )


df = load_dataset(input_path)

print("=" * 75)
print("UPSTREAM DATASET LOADED")
print("=" * 75)
print(f"Input file       : {input_path}")
print(f"Rows             : {len(df):,}")
print(f"Columns          : {len(df.columns):,}")
print("=" * 75)

if df.empty:
    raise ValueError("The upstream dataset is empty.")

df.head(3)

UPSTREAM DATASET LOADED
Input file       : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\unified_supply_data.json
Rows             : 303
Columns          : 137


,record_id,source,title,company,supplier,product,event,location,currency,url,availability_state,supplier_normalized,source_normalized,preliminary_risk_band,price,price_log,price_deviation_pct,price_robust_zscore,price_zscore,price_percentile,price_iqr_outlier,product_price_deviation_pct,supplier_price_deviation_pct,availability_risk_score,availability_text_length,availability_missing_flag,rating,rating_normalized,rating_risk_score,review_length,review_word_count,review_char_count,review_exclamation_count,review_question_count,review_uppercase_ratio,review_missing_flag,rating_missing_flag,review_quality_signal,disruption_keyword_count,negative_keyword_count,urgency_keyword_count,counterfeit_keyword_count,disruption_signal_flag,negative_signal_flag,urgency_signal_flag,counterfeit_signal_flag,text_risk_score,supplier_observation_count,supplier_median_price,supplier_mean_price,supplier_price_std,supplier_mean_rating,supplier_mean_availability_risk,supplier_mean_text_risk,supplier_disruption_frequency,supplier_negative_frequency,supplier_data_quality_risk,source_observation_count,source_mean_rating,source_mean_availability_risk,source_mean_text_risk,source_disruption_frequency,record_missingness_ratio,record_completeness_score,event_year,event_month,event_day,event_day_of_week,event_week_of_year,event_hour,month_sin,month_cos,day_of_week_sin,day_of_week_cos,preliminary_risk_signal,preliminary_risk_score,ml_anomaly_score,ml_anomaly_score_100,ml_anomaly_label,anomaly_severity,anomaly_reasons,review,availability,title_normalized,event_normalized,review_normalized,availability_normalized,company_normalized,product_normalized,location_normalized,nlp_document,nlp_document_length,nlp_document_word_count,nlp_supply_disruption_count,nlp_production_disruption_count,nlp_logistics_disruption_count,nlp_supplier_risk_count,nlp_quality_risk_count,nlp_counterfeit_risk_count,nlp_negative_sentiment_count,nlp_urgency_count,nlp_price_pressure_count,nlp_primary_risk_category,nlp_sentiment_compound,nlp_negative_sentiment_score,nlp_sentiment_label,nlp_exclamation_count,nlp_question_count,nlp_numeric_token_count,nlp_uppercase_ratio,nlp_urgency_score,nlp_disruption_score,nlp_supplier_risk_score,nlp_quality_risk_score,nlp_counterfeit_risk_score,nlp_price_pressure_score,nlp_risk_score,nlp_risk_score_100,nlp_risk_band,nlp_risk_reasons,ml_nlp_signal_gap,ml_nlp_combined_signal,ml_nlp_combined_score_100,ml_nlp_signal_status,_nlp_title,_nlp_event,_nlp_review,_nlp_supplier,_nlp_company,_nlp_product,_nlp_source,_nlp_location,nlp_supplier_risk_categories,nlp_supplier_risk_terms,nlp_supplier_risk_score_100,nlp_supplier_risk_band,nlp_supplier_risk_reason
0,SS-000001,DeoDap,Wall Mount Mop Holder – No-Slide Grip for Home & Garage | GlimmerHome,GlimmerHome,,Wall Mount Mop Holder – No-Slide Grip for Home & Garage,,,INR,https://deodap.in/products/hardware-tool-multifunction-wall-mount-garage-holder-for-mop-broom-hanger,unknown,,deodap,MINIMAL,190.00,5.25,-80.00,-0.04,-0.13,0.28,0,0.00,-80.00,0.50,0,1,4.60,0.92,0.08,249,42,249,0,0,0.00,0,0,3.76,0,0,0,0,0,0,0,0,0.00,303,950.00,"207,897.85","1,598,459.63",4.20,0.41,0.00,0.00,0.01,0.15,16,4.57,0.50,0.00,0.00,0.40,0.60,2026,8,19,2,34.00,18,-0.87,-0.50,0.97,-0.22,0.15,14.98,0.73,72.83,0,HIGH,price below median by 80.0%,,,wall mount mop holder no-slide grip for home garage glimmerhome,,,,glimmerhome,wall mount mop holder no-slide grip for home garage,,wall mount mop holder no-slide grip for home garage glimmerhome glimmerhome wall mount mop holder no-slide grip for ...,127,20,0,0,0,0,0,0,0,0,0,NORMAL,0.00,0.50,NEUTRAL,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.05,5.00,MINIMAL,no strong domain-specific textual risk signal,0.68,0.42,42.31,REVIEW_REQUIRED,wall mount mop holder – no-slide grip for home & garage | glimmerhome,,,,glimmerhome,wall mount mop holder – no-slide grip for home & garage,deodap,,[],[],0.00,LOW,No supplier-risk signal detected.
1,SS-000002,DeoDap,Compact TianMu Tool Set – Essential 9-Piece Repair Kit for Home,D

In [30]:
# ============================================================
# CELL 6 — ROBUST DATA QUALITY + DUPLICATE VALIDATION
# SupplyShield AI — Final Risk Scoring Engine
# ============================================================

import ast
import json
import numpy as np
import pandas as pd

print("=" * 80)
print("SUPPLYSHIELD AI — ROBUST DATA QUALITY VALIDATION")
print("=" * 80)


# ------------------------------------------------------------
# 1. Validate that the upstream dataframe exists
# ------------------------------------------------------------

if "df_risk" not in globals():

    raise RuntimeError(
        "df_risk is not available.\n"
        "Please run the upstream data-loading cell first."
    )


if not isinstance(df_risk, pd.DataFrame):

    raise TypeError(
        "df_risk must be a pandas DataFrame."
    )


if df_risk.empty:

    raise ValueError(
        "df_risk is empty. Expected the NLP pipeline "
        "to provide approximately 303 records."
    )


print("\nInput dataframe:")
print(
    f"Rows    : {len(df_risk)}"
)

print(
    f"Columns : {len(df_risk.columns)}"
)

print(
    f"Shape   : {df_risk.shape}"
)


# ------------------------------------------------------------
# 2. Create a defensive working copy
# ------------------------------------------------------------

risk_df = df_risk.copy()


# ------------------------------------------------------------
# 3. Normalize column names
# ------------------------------------------------------------

risk_df.columns = [
    str(column).strip()
    for column in risk_df.columns
]


# ------------------------------------------------------------
# 4. Detect complex/list/dictionary columns
# ------------------------------------------------------------

complex_columns = []

for column in risk_df.columns:

    sample_values = (
        risk_df[column]
        .dropna()
        .head(100)
        .tolist()
    )

    contains_complex_value = any(
        isinstance(
            value,
            (
                list,
                tuple,
                dict,
                set,
                np.ndarray
            )
        )
        for value in sample_values
    )

    if contains_complex_value:

        complex_columns.append(column)


print("\nComplex-value columns detected:")
print("-" * 80)

if complex_columns:

    for column in complex_columns:

        print(
            f"  • {column}"
        )

else:

    print("  None")


# ------------------------------------------------------------
# 5. Create a hash-safe dataframe ONLY for duplicate checking
# ------------------------------------------------------------

duplicate_check_df = risk_df.copy()


def make_hash_safe(value):
    """
    Convert nested Python structures into deterministic,
    hashable representations.

    IMPORTANT:
    This function is used ONLY for duplicate detection.
    The original risk_df remains untouched.
    """

    if isinstance(value, np.ndarray):

        return tuple(
            make_hash_safe(item)
            for item in value.tolist()
        )


    if isinstance(value, list):

        return tuple(
            make_hash_safe(item)
            for item in value
        )


    if isinstance(value, tuple):

        return tuple(
            make_hash_safe(item)
            for item in value
        )


    if isinstance(value, set):

        return tuple(
            sorted(
                (
                    make_hash_safe(item)
                    for item in value
                ),
                key=str
            )
        )


    if isinstance(value, dict):

        return tuple(
            sorted(
                (
                    str(key),
                    make_hash_safe(item)
                )
                for key, item in value.items()
            )
        )


    if isinstance(value, pd.Timestamp):

        return value.isoformat()


    if isinstance(value, np.generic):

        return value.item()


    return value


# ------------------------------------------------------------
# 6. Apply safe conversion
# ------------------------------------------------------------

for column in duplicate_check_df.columns:

    if column in complex_columns:

        duplicate_check_df[column] = (
            duplicate_check_df[column]
            .map(make_hash_safe)
        )


# ------------------------------------------------------------
# 7. Detect exact duplicate records
# ------------------------------------------------------------

try:

    duplicate_mask = (
        duplicate_check_df
        .duplicated(
            keep="first"
        )
    )

    duplicate_count = int(
        duplicate_mask.sum()
    )

except Exception as error:

    print(
        "\nWARNING: Standard duplicate detection "
        "failed unexpectedly."
    )

    print(
        "Reason:",
        repr(error)
    )

    duplicate_count = 0

    duplicate_mask = pd.Series(
        False,
        index=risk_df.index
    )


# ------------------------------------------------------------
# 8. Remove exact duplicates
# ------------------------------------------------------------

if duplicate_count > 0:

    risk_df = risk_df.loc[
        ~duplicate_mask
    ].copy()

    risk_df.reset_index(
        drop=True,
        inplace=True
    )


# ------------------------------------------------------------
# 9. Missing-value profile
# ------------------------------------------------------------

missing_summary = (
    risk_df
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)


missing_columns = (
    missing_summary[
        missing_summary > 0
    ]
)


print("\n")
print("=" * 80)
print("DATA QUALITY SUMMARY")
print("=" * 80)

print(
    f"Original records       : {len(df_risk)}"
)

print(
    f"Duplicate records      : {duplicate_count}"
)

print(
    f"Final records          : {len(risk_df)}"
)

print(
    f"Total columns          : {len(risk_df.columns)}"
)

print(
    f"Columns with missing   : {len(missing_columns)}"
)


# ------------------------------------------------------------
# 10. Display missing-value profile
# ------------------------------------------------------------

if len(missing_columns) > 0:

    print("\nTop missing-value columns:")
    print("-" * 80)

    for column, count in (
        missing_columns
        .head(20)
        .items()
    ):

        percentage = (
            count
            / len(risk_df)
            * 100
        )

        print(
            f"{column:<40}"
            f"{count:>8}"
            f" ({percentage:>6.2f}%)"
        )

else:

    print(
        "\nNo missing values detected."
    )


# ------------------------------------------------------------
# 11. Validate numeric columns
# ------------------------------------------------------------

numeric_columns = (
    risk_df
    .select_dtypes(
        include=np.number
    )
    .columns
    .tolist()
)


print("\nNumeric feature count:")
print(
    len(numeric_columns)
)


# ------------------------------------------------------------
# 12. Check for infinite numeric values
# ------------------------------------------------------------

infinite_summary = {}

for column in numeric_columns:

    values = pd.to_numeric(
        risk_df[column],
        errors="coerce"
    )

    infinite_count = int(
        np.isinf(values).sum()
    )

    if infinite_count > 0:

        infinite_summary[column] = (
            infinite_count
        )


if infinite_summary:

    print("\nInfinite numeric values detected:")

    for column, count in (
        infinite_summary.items()
    ):

        print(
            f"  {column}: {count}"
        )

else:

    print(
        "\nNo infinite numeric values detected."
    )


# ------------------------------------------------------------
# 13. Replace infinite values safely
# ------------------------------------------------------------

if infinite_summary:

    risk_df = risk_df.replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )


# ------------------------------------------------------------
# 14. Validate supplier information
# ------------------------------------------------------------

supplier_candidates = [
    "supplier",
    "company",
    "seller",
    "vendor"
]


supplier_column = None


for candidate in supplier_candidates:

    if candidate in risk_df.columns:

        supplier_column = candidate
        break


if supplier_column is not None:

    supplier_values = (
        risk_df[supplier_column]
        .astype("string")
        .str.strip()
    )

    supplier_values = (
        supplier_values
        .replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "None": pd.NA
            }
        )
    )

    unique_suppliers = int(
        supplier_values
        .dropna()
        .nunique()
    )

    missing_suppliers = int(
        supplier_values
        .isna()
        .sum()
    )

else:

    unique_suppliers = 0
    missing_suppliers = len(risk_df)


print("\nSupplier validation:")
print(
    f"Supplier column       : {supplier_column}"
)

print(
    f"Unique suppliers      : {unique_suppliers}"
)

print(
    f"Missing supplier rows : {missing_suppliers}"
)


# ------------------------------------------------------------
# 15. Identify NLP risk features
# ------------------------------------------------------------

nlp_columns = [
    column
    for column in risk_df.columns
    if str(column).lower().startswith("nlp_")
]


print("\nNLP feature inventory:")
print(
    f"NLP columns detected  : {len(nlp_columns)}"
)


if nlp_columns:

    for column in nlp_columns:

        print(
            f"  • {column}"
        )


# ------------------------------------------------------------
# 16. Validate risk-related columns
# ------------------------------------------------------------

risk_keywords = [
    "risk",
    "score",
    "anomaly",
    "sentiment",
    "disruption",
    "urgency",
    "pressure"
]


risk_feature_columns = [
    column
    for column in risk_df.columns
    if any(
        keyword in str(column).lower()
        for keyword in risk_keywords
    )
]


print("\nRisk-related features:")
print(
    f"Risk features detected : "
    f"{len(risk_feature_columns)}"
)


# ------------------------------------------------------------
# 17. Final integrity assertions
# ------------------------------------------------------------

assert len(risk_df) > 0, (
    "Final risk dataframe contains zero records."
)

assert len(risk_df.columns) > 0, (
    "Final risk dataframe contains zero columns."
)


# We expect the upstream pipeline to contain the original
# 303 records unless duplicates genuinely exist.

if len(df_risk) == 303:

    print(
        "\nUpstream record-count check:"
    )

    print(
        f"Expected upstream records : 303"
    )

    print(
        f"Received upstream records : {len(df_risk)}"
    )

    print(
        f"Final records after dedupe : {len(risk_df)}"
    )


# ------------------------------------------------------------
# 18. Preserve dataframe for downstream cells
# ------------------------------------------------------------

df_risk = risk_df.copy()


# ------------------------------------------------------------
# 19. Final status
# ------------------------------------------------------------

print("\n")
print("=" * 80)
print("DATA QUALITY VALIDATION — PASSED")
print("=" * 80)

print(
    f"Final dataframe shape : {df_risk.shape}"
)

print(
    f"Complex columns       : {len(complex_columns)}"
)

print(
    f"Duplicates removed    : {duplicate_count}"
)

print(
    f"Numeric features      : {len(numeric_columns)}"
)

print(
    f"NLP features          : {len(nlp_columns)}"
)

print(
    f"Risk features         : {len(risk_feature_columns)}"
)

print(
    f"Unique suppliers      : {unique_suppliers}"
)

print("=" * 80)

print(
    "\nREADY FOR RISK SIGNAL ENGINEERING."
)

SUPPLYSHIELD AI — ROBUST DATA QUALITY VALIDATION

Input dataframe:
Rows    : 303
Columns : 147
Shape   : (303, 147)

Complex-value columns detected:
--------------------------------------------------------------------------------
  • nlp_supplier_risk_categories
  • nlp_supplier_risk_terms


DATA QUALITY SUMMARY
Original records       : 303
Duplicate records      : 0
Final records          : 303
Total columns          : 147
Columns with missing   : 1

Top missing-value columns:
--------------------------------------------------------------------------------
currency                                      48 ( 15.84%)

Numeric feature count:
102

No infinite numeric values detected.

Supplier validation:
Supplier column       : supplier
Unique suppliers      : 0
Missing supplier rows : 303

NLP feature inventory:
NLP columns detected  : 35
  • nlp_document
  • nlp_document_length
  • nlp_document_word_count
  • nlp_supply_disruption_count
  • nlp_production_disruption_count
  • nlp_logist

## 2. Normalize Upstream Risk Signals

Different upstream modules may produce different names, scales, or missing values. The scoring engine standardizes them to a common 0–100 scale before aggregation.

In [31]:
# ============================================================
# CELL 8 — Risk Signal Utility Functions
# ============================================================

def safe_numeric_series(
    series: pd.Series,
    default: float = 0.0
) -> pd.Series:
    """
    Convert arbitrary values to numeric safely.
    Invalid values become NaN and are replaced by default.
    """
    
    numeric = pd.to_numeric(
        series,
        errors="coerce"
    )

    return numeric.fillna(default).astype(float)


def clip_0_100(
    series: pd.Series
) -> pd.Series:
    """
    Constrain a numeric risk signal to [0, 100].
    """
    
    numeric = safe_numeric_series(series)

    return numeric.clip(
        lower=0,
        upper=100
    )


def normalize_risk_signal(
    series: pd.Series,
    source_scale: Optional[Tuple[float, float]] = None
) -> pd.Series:
    """
    Normalize an arbitrary risk signal into [0, 100].

    If source_scale is supplied, linear scaling is used.
    Otherwise values already in [0,100] are retained.
    """

    numeric = safe_numeric_series(series)

    if source_scale is not None:

        low, high = source_scale

        if high <= low:
            raise ValueError(
                "Invalid source scale."
            )

        normalized = (
            (numeric - low) /
            (high - low)
        ) * 100.0

        return normalized.clip(0, 100)

    # Handle probabilities.
    if numeric.max() <= 1.0:
        return (numeric * 100.0).clip(0, 100)

    return numeric.clip(0, 100)


def find_column(
    dataframe: pd.DataFrame,
    aliases: List[str]
) -> Optional[str]:
    """
    Find a column using exact or normalized matching.
    """

    normalized_map = {
        re.sub(r"[^a-z0-9]", "", str(col).lower()): col
        for col in dataframe.columns
    }

    # Exact normalized matching
    for alias in aliases:

        key = re.sub(
            r"[^a-z0-9]",
            "",
            alias.lower()
        )

        if key in normalized_map:
            return normalized_map[key]

    # Partial matching
    for alias in aliases:

        key = re.sub(
            r"[^a-z0-9]",
            "",
            alias.lower()
        )

        for normalized_col, original_col in normalized_map.items():

            if key in normalized_col:
                return original_col

    return None


print("Risk utility functions initialized successfully.")

Risk utility functions initialized successfully.


In [32]:
# ============================================================
# CELL 9 — Detect Available Upstream Risk Signals
# ============================================================

SIGNAL_ALIASES = {

    "nlp_risk": [
        "nlp_risk_score_100",
        "nlp_risk_score",
        "nlp_combined_risk",
        "nlp_risk"
    ],

    "supplier_risk": [
        "nlp_supplier_risk_score",
        "supplier_risk_score",
        "supplier_risk"
    ],

    "quality_risk": [
        "nlp_quality_risk_score",
        "quality_risk_score",
        "quality_risk"
    ],

    "counterfeit_risk": [
        "nlp_counterfeit_risk_score",
        "counterfeit_risk_score",
        "counterfeit_risk"
    ],

    "price_pressure": [
        "nlp_price_pressure_score",
        "price_pressure_score",
        "price_pressure"
    ],

    "urgency": [
        "nlp_urgency_score",
        "urgency_score",
        "urgency"
    ],

    "anomaly_risk": [
        "anomaly_risk_score",
        "ml_anomaly_risk_score",
        "anomaly_score",
        "anomaly_risk",
        "anomaly"
    ],

    "market_risk": [
        "market_risk_score",
        "market_risk",
        "ml_market_risk_score"
    ],

    "disruption_risk": [
        "disruption_risk_score",
        "ml_disruption_risk_score",
        "disruption_score",
        "disruption_risk"
    ],

    "webshield_risk": [
        "webshield_risk_score",
        "webshield_score",
        "counterfeit_risk_score"
    ],

    "combined_signal": [
        "ml_nlp_combined_signal",
        "combined_risk_signal",
        "combined_signal"
    ]
}


detected_signals = {}

print("=" * 75)
print("UPSTREAM SIGNAL DETECTION")
print("=" * 75)

for signal_name, aliases in SIGNAL_ALIASES.items():

    matched = find_column(
        df,
        aliases
    )

    detected_signals[signal_name] = matched

    status = "FOUND" if matched else "NOT FOUND"

    print(
        f"{signal_name:<22} : "
        f"{status:<10} "
        f"{matched if matched else '-'}"
    )

print("=" * 75)

UPSTREAM SIGNAL DETECTION
nlp_risk               : FOUND      nlp_risk_score_100
supplier_risk          : FOUND      nlp_supplier_risk_score
quality_risk           : FOUND      nlp_quality_risk_score
counterfeit_risk       : FOUND      nlp_counterfeit_risk_score
price_pressure         : FOUND      nlp_price_pressure_score
urgency                : FOUND      nlp_urgency_score
anomaly_risk           : FOUND      ml_anomaly_score
market_risk            : NOT FOUND  -
disruption_risk        : FOUND      nlp_disruption_score
webshield_risk         : FOUND      nlp_counterfeit_risk_score
combined_signal        : FOUND      ml_nlp_combined_signal


In [33]:
# ============================================================
# CELL 10 — Standardized Risk Signal Matrix
# ============================================================

risk_df = df.copy()

signal_columns_created = []

for signal_name, source_column in detected_signals.items():

    target_column = f"risk_{signal_name}"

    if source_column is None:
        risk_df[target_column] = 0.0

    else:
        risk_df[target_column] = normalize_risk_signal(
            risk_df[source_column]
        )

    signal_columns_created.append(
        target_column
    )


# ------------------------------------------------------------
# Remove duplicate information from combined NLP signal
# when the main NLP score already exists.
# ------------------------------------------------------------

if "risk_nlp_risk" in risk_df.columns:

    # The main NLP risk signal is authoritative.
    # Combined signal remains available for diagnostics.
    pass


print("=" * 75)
print("STANDARDIZED RISK SIGNAL MATRIX")
print("=" * 75)

signal_summary = []

for column in signal_columns_created:

    series = risk_df[column]

    signal_summary.append({
        "signal": column,
        "mean": round(float(series.mean()), 3),
        "median": round(float(series.median()), 3),
        "min": round(float(series.min()), 3),
        "max": round(float(series.max()), 3),
        "non_zero": int((series > 0).sum()),
        "missing": int(series.isna().sum())
    })


signal_summary_df = pd.DataFrame(
    signal_summary
)

display(signal_summary_df)

STANDARDIZED RISK SIGNAL MATRIX


,signal,mean,median,min,max,non_zero,missing
0,risk_nlp_risk,4.73,5.00,0.21,9.79,303,0
1,risk_supplier_risk,0.00,0.00,0.00,0.00,0,0
2,risk_quality_risk,0.00,0.00,0.00,0.00,0,0
3,risk_counterfeit_risk,0.00,0.00,0.00,0.00,0,0
4,risk_price_pressure,0.33,0.00,0.00,50.00,2,0
5,risk_urgency,0.00,0.00,0.00,0.00,0,0
6,risk_anomaly_risk,20.17,10.69,0.00,100.00,302,0
7,risk_market_risk,0.00,0.00,0.00,0.00,0,0
8,risk_disruption_risk,0.00,0.00,0.00,0.00,0,0
9,risk_webshield_risk,0.00,0.00,0.00,0.00,0,0


## 3. Configure the Explainable Risk Model

The final score combines multiple independent risk dimensions. Weights are configurable so the scoring policy can evolve without rewriting the pipeline.

In [34]:
# ============================================================
# CELL 12 — Production Risk Weight Configuration
# ============================================================

RISK_WEIGHTS = {
    "risk_nlp_risk": 0.20,
    "risk_supplier_risk": 0.15,
    "risk_quality_risk": 0.10,
    "risk_counterfeit_risk": 0.15,
    "risk_price_pressure": 0.10,
    "risk_urgency": 0.05,
    "risk_anomaly_risk": 0.10,
    "risk_market_risk": 0.05,
    "risk_disruption_risk": 0.05,
    "risk_webshield_risk": 0.05
}


# ------------------------------------------------------------
# Validate configuration
# ------------------------------------------------------------

weight_sum = sum(RISK_WEIGHTS.values())

if not math.isclose(
    weight_sum,
    1.0,
    abs_tol=1e-9
):
    raise ValueError(
        f"Risk weights must sum to 1.0. "
        f"Current sum = {weight_sum:.6f}"
    )


print("=" * 75)
print("RISK MODEL CONFIGURATION")
print("=" * 75)

for signal, weight in RISK_WEIGHTS.items():
    print(
        f"{signal:<30} : "
        f"{weight * 100:>6.2f}%"
    )

print("-" * 75)
print(
    f"{'TOTAL':<30} : "
    f"{weight_sum * 100:>6.2f}%"
)
print("=" * 75)

RISK MODEL CONFIGURATION
risk_nlp_risk                  :  20.00%
risk_supplier_risk             :  15.00%
risk_quality_risk              :  10.00%
risk_counterfeit_risk          :  15.00%
risk_price_pressure            :  10.00%
risk_urgency                   :   5.00%
risk_anomaly_risk              :  10.00%
risk_market_risk               :   5.00%
risk_disruption_risk           :   5.00%
risk_webshield_risk            :   5.00%
---------------------------------------------------------------------------
TOTAL                          : 100.00%


In [35]:
# ============================================================
# CELL 13 — Weighted Multi-Signal Risk Aggregation
# ============================================================

weighted_components = []

for signal_column, weight in RISK_WEIGHTS.items():

    component_column = (
        signal_column.replace(
            "risk_",
            "component_",
            1
        )
    )

    risk_df[component_column] = (
        risk_df[signal_column] * weight
    )

    weighted_components.append(
        component_column
    )


risk_df["final_risk_score_raw"] = (
    risk_df[weighted_components]
    .sum(axis=1)
)


# ------------------------------------------------------------
# Normalize final score to 0–100.
# ------------------------------------------------------------

risk_df["final_risk_score"] = (
    risk_df["final_risk_score_raw"]
    .clip(0, 100)
    .round(2)
)


print("=" * 75)
print("FINAL RISK SCORE GENERATED")
print("=" * 75)

print(
    risk_df["final_risk_score"]
    .describe()
)

print("=" * 75)

FINAL RISK SCORE GENERATED
count   303.00
mean      3.00
std       2.40
min       0.24
25%       1.36
50%       2.07
75%       3.67
max      11.45
Name: final_risk_score, dtype: float64


In [36]:
# ============================================================
# CELL 14 — Risk Band Classification
# ============================================================

def classify_risk(score: float) -> str:
    """
    Convert a 0–100 score into an operational risk band.
    """

    try:
        score = float(score)
    except (TypeError, ValueError):
        return "UNKNOWN"

    if score >= 80:
        return "CRITICAL"

    elif score >= 60:
        return "HIGH"

    elif score >= 35:
        return "MEDIUM"

    else:
        return "LOW"


risk_df["risk_band"] = (
    risk_df["final_risk_score"]
    .apply(classify_risk)
)


risk_distribution = (
    risk_df["risk_band"]
    .value_counts()
    .reindex(
        ["CRITICAL", "HIGH", "MEDIUM", "LOW"],
        fill_value=0
    )
)


print("=" * 75)
print("RISK BAND DISTRIBUTION")
print("=" * 75)

for band, count in risk_distribution.items():

    percentage = (
        count / len(risk_df) * 100
        if len(risk_df)
        else 0
    )

    print(
        f"{band:<10} : "
        f"{count:>5,} records "
        f"({percentage:>6.2f}%)"
    )

print("=" * 75)

RISK BAND DISTRIBUTION
CRITICAL   :     0 records (  0.00%)
HIGH       :     0 records (  0.00%)
MEDIUM     :     0 records (  0.00%)
LOW        :   303 records (100.00%)


In [37]:
# ============================================================
# CELL 15 — Explainable Risk Reason Generation
# ============================================================

REASON_THRESHOLDS = {
    "risk_nlp_risk": (
        60,
        "Strong NLP-derived risk signals"
    ),

    "risk_supplier_risk": (
        60,
        "Elevated supplier risk indicators"
    ),

    "risk_quality_risk": (
        60,
        "Potential quality-related risk"
    ),

    "risk_counterfeit_risk": (
        60,
        "Potential counterfeit/product-authenticity risk"
    ),

    "risk_price_pressure": (
        60,
        "Significant price-pressure signal"
    ),

    "risk_urgency": (
        60,
        "High urgency detected in source intelligence"
    ),

    "risk_anomaly_risk": (
        60,
        "Unusual/anomalous behaviour detected"
    ),

    "risk_market_risk": (
        60,
        "Elevated market-risk indicators"
    ),

    "risk_disruption_risk": (
        60,
        "Potential supply-chain disruption detected"
    ),

    "risk_webshield_risk": (
        60,
        "Elevated WebShield risk indicators"
    )
}


def generate_risk_reasons(row: pd.Series) -> List[str]:
    """
    Generate human-readable explanations based on
    individual risk dimensions.
    """

    reasons = []

    for signal, (threshold, message) in REASON_THRESHOLDS.items():

        value = row.get(
            signal,
            0
        )

        try:
            value = float(value)
        except (TypeError, ValueError):
            continue

        if value >= threshold:
            reasons.append(
                f"{message} ({value:.1f}/100)"
            )

    # Always provide a fallback explanation.
    if not reasons:

        reasons.append(
            "No individual risk signal exceeded the alert threshold."
        )

    return reasons


risk_df["risk_reasons"] = risk_df.apply(
    generate_risk_reasons,
    axis=1
)


# Human-readable version for exports.
risk_df["risk_reasons_text"] = (
    risk_df["risk_reasons"]
    .apply(
        lambda x: " | ".join(x)
        if isinstance(x, list)
        else str(x)
    )
)


print("Explainable risk reasons generated successfully.")

Explainable risk reasons generated successfully.


In [38]:
# ============================================================
# CELL 16 — Risk Confidence & Data Quality
# ============================================================

active_signal_columns = list(
    RISK_WEIGHTS.keys()
)


def calculate_confidence(row: pd.Series) -> float:
    """
    Estimate confidence based on availability of upstream
    signals. This is NOT probability of correctness.
    It measures data coverage.
    """

    available = 0.0
    total_weight = 0.0

    for signal in active_signal_columns:

        weight = RISK_WEIGHTS[signal]
        total_weight += weight

        value = row.get(
            signal,
            np.nan
        )

        try:
            numeric = float(value)
        except (TypeError, ValueError):
            numeric = np.nan

        if np.isfinite(numeric):
            available += weight

    if total_weight == 0:
        return 0.0

    return round(
        (available / total_weight) * 100,
        2
    )


risk_df["data_coverage_score"] = risk_df.apply(
    calculate_confidence,
    axis=1
)


def confidence_band(score: float) -> str:

    if score >= 90:
        return "HIGH"

    elif score >= 70:
        return "MEDIUM"

    elif score >= 40:
        return "LOW"

    return "VERY_LOW"


risk_df["confidence_band"] = (
    risk_df["data_coverage_score"]
    .apply(confidence_band)
)


print("=" * 75)
print("DATA COVERAGE / CONFIDENCE SUMMARY")
print("=" * 75)

print(
    risk_df["data_coverage_score"]
    .describe()
)

print("\nConfidence bands:")
print(
    risk_df["confidence_band"]
    .value_counts()
)

DATA COVERAGE / CONFIDENCE SUMMARY
count   303.00
mean    100.00
std       0.00
min     100.00
25%     100.00
50%     100.00
75%     100.00
max     100.00
Name: data_coverage_score, dtype: float64

Confidence bands:
confidence_band
HIGH    303
Name: count, dtype: int64


In [39]:
# ============================================================
# CELL 17 — Supplier-Level Risk Aggregation
# ============================================================

supplier_column = find_column(
    risk_df,
    [
        "supplier",
        "supplier_name",
        "vendor",
        "seller"
    ]
)

company_column = find_column(
    risk_df,
    [
        "company",
        "organization",
        "organisation"
    ]
)


if supplier_column:

    supplier_risk_summary = (
        risk_df
        .groupby(
            supplier_column,
            dropna=False
        )
        .agg(
            records=("final_risk_score", "count"),
            avg_risk=("final_risk_score", "mean"),
            max_risk=("final_risk_score", "max"),
            min_risk=("final_risk_score", "min"),
            avg_confidence=("data_coverage_score", "mean")
        )
        .reset_index()
    )

    supplier_risk_summary["avg_risk"] = (
        supplier_risk_summary["avg_risk"]
        .round(2)
    )

    supplier_risk_summary["max_risk"] = (
        supplier_risk_summary["max_risk"]
        .round(2)
    )

    supplier_risk_summary["min_risk"] = (
        supplier_risk_summary["min_risk"]
        .round(2)
    )

    supplier_risk_summary["avg_confidence"] = (
        supplier_risk_summary["avg_confidence"]
        .round(2)
    )

    supplier_risk_summary["risk_band"] = (
        supplier_risk_summary["avg_risk"]
        .apply(classify_risk)
    )

    supplier_risk_summary = (
        supplier_risk_summary
        .sort_values(
            ["avg_risk", "max_risk"],
            ascending=False
        )
        .reset_index(drop=True)
    )

    print("=" * 75)
    print("SUPPLIER-LEVEL RISK SUMMARY")
    print("=" * 75)

    display(
        supplier_risk_summary.head(25)
    )

else:

    supplier_risk_summary = pd.DataFrame()

    print(
        "Supplier column not found. "
        "Supplier aggregation skipped."
    )

SUPPLIER-LEVEL RISK SUMMARY


,supplier,records,avg_risk,max_risk,min_risk,avg_confidence,risk_band
0,,303,3.00,11.45,0.24,100.00,LOW


In [40]:
# ============================================================
# CELL 18 — Enterprise Risk Ranking
# ============================================================

risk_df["risk_rank"] = (
    risk_df["final_risk_score"]
    .rank(
        method="dense",
        ascending=False
    )
    .astype(int)
)


top_risk_columns = [
    col
    for col in [
        "source",
        "title",
        "supplier",
        "company",
        "product",
        "event",
        "location",
        "final_risk_score",
        "risk_band",
        "data_coverage_score",
        "confidence_band",
        "risk_reasons_text"
    ]
    if col in risk_df.columns
]


top_risk_records = (
    risk_df[
        top_risk_columns
    ]
    .sort_values(
        "final_risk_score",
        ascending=False
    )
    .head(25)
    .reset_index(drop=True)
)


print("=" * 75)
print("TOP 25 HIGH-RISK RECORDS")
print("=" * 75)

display(top_risk_records)

TOP 25 HIGH-RISK RECORDS


,source,title,supplier,company,product,event,location,final_risk_score,risk_band,data_coverage_score,confidence_band,risk_reasons_text
0,DeoDap,Leak Proof Tape – Instant Waterproof Seal for Repairs | BoltForce,,BoltForce,Leak Proof Tape – Instant Waterproof Seal for Repairs,,,11.45,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (98.7/100)
1,TradeIndia,Siemens Artis Zee Cath Lab - Application: Hospital,,Arihant Medmach Private Limited,Siemens Artis Zee Cath Lab,,,11.13,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (99.0/100)
2,TradeIndia,Allenger Altima F100 Fixed Cath Lab Machine,,Rentomed,Allenger Altima F100 Fixed Cath Lab Machine,,,11.00,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (100.0/100)
3,TradeIndia,Cath Lab Machine,,Mf India,Cath Lab Machine,,,10.67,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (96.7/100)
4,TradeIndia,Broken Bag Detector,,Toshbro Controls Pvt. Ltd.,Broken Bag Detector,,,9.88,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (81.5/100)
5,DeoDap,Manual Wall Fastening Nail Gun Tool for Wood and Concrete Walls (1 Set),,DeoDap,Manual Wall Fastening Nail Gun Tool for Wood and Concrete Walls (1 Set),,,9.70,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (81.1/100)
6,DeoDap,Manual Wall Fastening Nail Gun Tool Set (1 Set),,DeoDap,Manual Wall Fastening Nail Gun Tool Set (1 Set),,,9.66,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (80.7/100)
7,TradeIndia,Broken Bag Detector - Accuracy: +2 %,,Applied Techno Engineers Private Limited,Broken Bag Detector,,,9.62,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (78.3/100)
8,TradeIndia,"Broken Bag Detector - 315 Grade Stainless Steel, 115x65x55 Mm | Adjustable Sensitivity, Instantaneous Response, Remo...",,Hnl Systems Pvt. Ltd.,Broken Bag Detector,,,9.49,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (77.6/100)
9,DeoDap,Hardware Tool Set – 11 Pcs Multi-Functional Kit | BoltForce,,BoltForce,Hardware Tool Set – 11 Pcs Multi-Functional Kit,,,9.48,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (84.8/100)


## 4. Final Risk Engine Diagnostics

The following validation checks confirm score ranges, distribution, missing values, and consistency before the dataset is exported.

In [41]:
# ============================================================
# CELL 20 — Production Validation Suite
# ============================================================

validation_results = {}


# ------------------------------------------------------------
# Check 1 — Dataset not empty
# ------------------------------------------------------------

validation_results["dataset_non_empty"] = (
    len(risk_df) > 0
)


# ------------------------------------------------------------
# Check 2 — Final score exists
# ------------------------------------------------------------

validation_results["final_score_exists"] = (
    "final_risk_score" in risk_df.columns
)


# ------------------------------------------------------------
# Check 3 — Score range
# ------------------------------------------------------------

validation_results["score_range_valid"] = (
    risk_df["final_risk_score"]
    .between(0, 100)
    .all()
)


# ------------------------------------------------------------
# Check 4 — No infinite scores
# ------------------------------------------------------------

validation_results["no_infinite_scores"] = (
    np.isfinite(
        risk_df["final_risk_score"]
    ).all()
)


# ------------------------------------------------------------
# Check 5 — Risk bands valid
# ------------------------------------------------------------

valid_bands = {
    "LOW",
    "MEDIUM",
    "HIGH",
    "CRITICAL"
}

validation_results["risk_bands_valid"] = (
    set(
        risk_df["risk_band"].dropna().unique()
    ).issubset(valid_bands)
)


# ------------------------------------------------------------
# Check 6 — Weight integrity
# ------------------------------------------------------------

validation_results["weights_sum_to_one"] = (
    math.isclose(
        sum(RISK_WEIGHTS.values()),
        1.0,
        abs_tol=1e-9
    )
)


# ------------------------------------------------------------
# Check 7 — Row count preserved
# ------------------------------------------------------------

validation_results["row_count_preserved"] = (
    len(risk_df) == len(df)
)


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("=" * 75)
print("PRODUCTION VALIDATION")
print("=" * 75)

all_passed = True

for check, passed in validation_results.items():

    status = "PASS" if passed else "FAIL"

    print(
        f"{check:<35} : {status}"
    )

    if not passed:
        all_passed = False


print("-" * 75)

if all_passed:
    print("ALL VALIDATION CHECKS PASSED.")
else:
    print("WARNING: ONE OR MORE VALIDATION CHECKS FAILED.")

print("=" * 75)

PRODUCTION VALIDATION
dataset_non_empty                   : PASS
final_score_exists                  : PASS
score_range_valid                   : PASS
no_infinite_scores                  : PASS
risk_bands_valid                    : PASS
weights_sum_to_one                  : PASS
row_count_preserved                 : PASS
---------------------------------------------------------------------------
ALL VALIDATION CHECKS PASSED.


In [42]:
# ============================================================
# CELL 21 — Final Risk Statistics
# ============================================================

risk_statistics = {
    "total_records": int(len(risk_df)),

    "average_risk_score": round(
        float(
            risk_df["final_risk_score"].mean()
        ),
        2
    ),

    "median_risk_score": round(
        float(
            risk_df["final_risk_score"].median()
        ),
        2
    ),

    "maximum_risk_score": round(
        float(
            risk_df["final_risk_score"].max()
        ),
        2
    ),

    "minimum_risk_score": round(
        float(
            risk_df["final_risk_score"].min()
        ),
        2
    ),

    "critical_records": int(
        (risk_df["risk_band"] == "CRITICAL").sum()
    ),

    "high_records": int(
        (risk_df["risk_band"] == "HIGH").sum()
    ),

    "medium_records": int(
        (risk_df["risk_band"] == "MEDIUM").sum()
    ),

    "low_records": int(
        (risk_df["risk_band"] == "LOW").sum()
    ),

    "average_data_coverage": round(
        float(
            risk_df["data_coverage_score"].mean()
        ),
        2
    )
}


print("=" * 75)
print("FINAL RISK ENGINE STATISTICS")
print("=" * 75)

for key, value in risk_statistics.items():

    label = key.replace(
        "_",
        " "
    ).title()

    print(
        f"{label:<30}: {value}"
    )

print("=" * 75)

FINAL RISK ENGINE STATISTICS
Total Records                 : 303
Average Risk Score            : 3.0
Median Risk Score             : 2.07
Maximum Risk Score            : 11.45
Minimum Risk Score            : 0.24
Critical Records              : 0
High Records                  : 0
Medium Records                : 0
Low Records                   : 303
Average Data Coverage         : 100.0


In [43]:
# ============================================================
# CELL 22 — Production Dataset Preparation
# ============================================================

final_df = risk_df.copy()


# ------------------------------------------------------------
# Convert list-based JSON fields to serializable strings.
# ------------------------------------------------------------

for column in final_df.columns:

    if final_df[column].dtype == "object":

        final_df[column] = final_df[column].apply(
            lambda value: (
                " | ".join(map(str, value))
                if isinstance(value, list)
                else value
            )
        )


# ------------------------------------------------------------
# Add pipeline metadata
# ------------------------------------------------------------

final_df["risk_engine_version"] = "1.0.0"

final_df["risk_engine_timestamp"] = (
    datetime.now(
        timezone.utc
    ).isoformat()
)


print("=" * 75)
print("FINAL DATASET PREPARED")
print("=" * 75)

print(
    f"Rows              : {len(final_df):,}"
)

print(
    f"Columns            : {len(final_df.columns):,}"
)

print(
    f"Risk score range   : "
    f"{final_df['final_risk_score'].min():.2f} – "
    f"{final_df['final_risk_score'].max():.2f}"
)

print("=" * 75)

FINAL DATASET PREPARED
Rows              : 303
Columns            : 168
Risk score range   : 0.24 – 11.45


In [44]:
# ============================================================
# CELL 23 — Export Final JSON Dataset
# ============================================================

final_json_path = (
    DATA_OUTPUT /
    "final_risk_scored_supply_data.json"
)


records = final_df.to_dict(
    orient="records"
)


with open(
    final_json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=2,
        default=str
    )


print("=" * 75)
print("JSON EXPORT COMPLETED")
print("=" * 75)
print(
    f"File : {final_json_path}"
)
print(
    f"Size : "
    f"{final_json_path.stat().st_size / 1024:.2f} KB"
)
print("=" * 75)

JSON EXPORT COMPLETED
File : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\risk_scoring\final_risk_scored_supply_data.json
Size : 1994.80 KB


In [45]:
# ============================================================
# CELL 24 — Export Final CSV Dataset
# ============================================================

final_csv_path = (
    DATA_OUTPUT /
    "final_risk_scored_supply_data.csv"
)


final_df.to_csv(
    final_csv_path,
    index=False,
    encoding="utf-8-sig"
)


print("=" * 75)
print("CSV EXPORT COMPLETED")
print("=" * 75)
print(
    f"File : {final_csv_path}"
)
print(
    f"Size : "
    f"{final_csv_path.stat().st_size / 1024:.2f} KB"
)
print("=" * 75)

CSV EXPORT COMPLETED
File : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\risk_scoring\final_risk_scored_supply_data.csv
Size : 554.33 KB


In [46]:
# ============================================================
# CELL 25 — Export Supplier Risk Intelligence
# ============================================================

supplier_output_path = (
    DATA_OUTPUT /
    "supplier_risk_summary.csv"
)


if not supplier_risk_summary.empty:

    supplier_risk_summary.to_csv(
        supplier_output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("=" * 75)
    print("SUPPLIER RISK EXPORT COMPLETED")
    print("=" * 75)
    print(
        f"File : {supplier_output_path}"
    )
    print(
        f"Suppliers : "
        f"{len(supplier_risk_summary):,}"
    )
    print("=" * 75)

else:

    print(
        "Supplier summary was not exported "
        "because no supplier field was detected."
    )

SUPPLIER RISK EXPORT COMPLETED
File : c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\risk_scoring\supplier_risk_summary.csv
Suppliers : 1


In [47]:
# ============================================================
# CELL 26 — Export Risk Engine Configuration
# ============================================================

config_path = (
    DATA_OUTPUT /
    "risk_engine_config.json"
)


configuration = {

    "engine": {
        "name": "SupplyShield AI Final Risk Scoring Engine",
        "version": "1.0.0",
        "scoring_range": [0, 100]
    },

    "risk_bands": {
        "LOW": "0–34.99",
        "MEDIUM": "35–59.99",
        "HIGH": "60–79.99",
        "CRITICAL": "80–100"
    },

    "weights": RISK_WEIGHTS,

    "input_dataset": str(
        input_path
    ),

    "output_directory": str(
        DATA_OUTPUT
    ),

    "generated_at": datetime.now(
        timezone.utc
    ).isoformat()
}


with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        configuration,
        f,
        indent=2
    )


print("=" * 75)
print("RISK ENGINE CONFIGURATION SAVED")
print("=" * 75)
print(config_path)
print("=" * 75)

RISK ENGINE CONFIGURATION SAVED
c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\risk_scoring\risk_engine_config.json


In [48]:
# ============================================================
# CELL 27 — Executive Risk Intelligence Output
# ============================================================

print()
print("=" * 85)
print("                    SUPPLYSHIELD AI")
print("               FINAL RISK INTELLIGENCE")
print("=" * 85)

print(
    f"Records analysed       : "
    f"{len(final_df):,}"
)

print(
    f"Average risk           : "
    f"{final_df['final_risk_score'].mean():.2f}/100"
)

print(
    f"Maximum risk           : "
    f"{final_df['final_risk_score'].max():.2f}/100"
)

print(
    f"Critical alerts        : "
    f"{(final_df['risk_band'] == 'CRITICAL').sum():,}"
)

print(
    f"High-risk alerts       : "
    f"{(final_df['risk_band'] == 'HIGH').sum():,}"
)

print(
    f"Medium-risk records    : "
    f"{(final_df['risk_band'] == 'MEDIUM').sum():,}"
)

print(
    f"Low-risk records       : "
    f"{(final_df['risk_band'] == 'LOW').sum():,}"
)

print(
    f"Average data coverage  : "
    f"{final_df['data_coverage_score'].mean():.2f}%"
)

print("-" * 85)

print("OUTPUT FILES")
print(f"1. {final_json_path}")
print(f"2. {final_csv_path}")
print(f"3. {supplier_output_path if not supplier_risk_summary.empty else 'N/A'}")
print(f"4. {config_path}")

print("=" * 85)
print("FINAL RISK SCORING ENGINE COMPLETED SUCCESSFULLY")
print("=" * 85)


                    SUPPLYSHIELD AI
               FINAL RISK INTELLIGENCE
Records analysed       : 303
Average risk           : 3.00/100
Maximum risk           : 11.45/100
Critical alerts        : 0
High-risk alerts       : 0
Medium-risk records    : 0
Low-risk records       : 303
Average data coverage  : 100.00%
-------------------------------------------------------------------------------------
OUTPUT FILES
1. c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\risk_scoring\final_risk_scored_supply_data.json
2. c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\risk_scoring\final_risk_scored_supply_data.csv
3. c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\risk_scoring\supplier_risk_summary.csv
4. c:\Users\Pradeep Bhat\Documents\Hackathons\SupplyShield-AI\member2\data\processed\risk_scoring\risk_engine_config.json
FINAL RISK SCORING ENGINE COMPLETED SUCCESSFULLY


In [49]:
# ============================================================
# CELL 28 — Dashboard/API Ready Sample
# ============================================================

dashboard_columns = [
    column
    for column in [
        "source",
        "title",
        "supplier",
        "company",
        "product",
        "event",
        "location",
        "final_risk_score",
        "risk_band",
        "data_coverage_score",
        "confidence_band",
        "risk_reasons_text"
    ]
    if column in final_df.columns
]


dashboard_ready_df = (
    final_df[
        dashboard_columns
    ]
    .sort_values(
        "final_risk_score",
        ascending=False
    )
    .reset_index(drop=True)
)


print("=" * 75)
print("DASHBOARD-READY DATA")
print("=" * 75)

display(
    dashboard_ready_df.head(10)
)

DASHBOARD-READY DATA


,source,title,supplier,company,product,event,location,final_risk_score,risk_band,data_coverage_score,confidence_band,risk_reasons_text
0,DeoDap,Leak Proof Tape – Instant Waterproof Seal for Repairs | BoltForce,,BoltForce,Leak Proof Tape – Instant Waterproof Seal for Repairs,,,11.45,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (98.7/100)
1,TradeIndia,Siemens Artis Zee Cath Lab - Application: Hospital,,Arihant Medmach Private Limited,Siemens Artis Zee Cath Lab,,,11.13,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (99.0/100)
2,TradeIndia,Allenger Altima F100 Fixed Cath Lab Machine,,Rentomed,Allenger Altima F100 Fixed Cath Lab Machine,,,11.00,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (100.0/100)
3,TradeIndia,Cath Lab Machine,,Mf India,Cath Lab Machine,,,10.67,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (96.7/100)
4,TradeIndia,Broken Bag Detector,,Toshbro Controls Pvt. Ltd.,Broken Bag Detector,,,9.88,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (81.5/100)
5,DeoDap,Manual Wall Fastening Nail Gun Tool for Wood and Concrete Walls (1 Set),,DeoDap,Manual Wall Fastening Nail Gun Tool for Wood and Concrete Walls (1 Set),,,9.70,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (81.1/100)
6,DeoDap,Manual Wall Fastening Nail Gun Tool Set (1 Set),,DeoDap,Manual Wall Fastening Nail Gun Tool Set (1 Set),,,9.66,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (80.7/100)
7,TradeIndia,Broken Bag Detector - Accuracy: +2 %,,Applied Techno Engineers Private Limited,Broken Bag Detector,,,9.62,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (78.3/100)
8,TradeIndia,"Broken Bag Detector - 315 Grade Stainless Steel, 115x65x55 Mm | Adjustable Sensitivity, Instantaneous Response, Remo...",,Hnl Systems Pvt. Ltd.,Broken Bag Detector,,,9.49,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (77.6/100)
9,DeoDap,Hardware Tool Set – 11 Pcs Multi-Functional Kit | BoltForce,,BoltForce,Hardware Tool Set – 11 Pcs Multi-Functional Kit,,,9.48,LOW,100.00,HIGH,Unusual/anomalous behaviour detected (84.8/100)
